[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rsasaki0109/slam-handbook-python/blob/main/part1_foundations/ch01_factor_graphs/03_sparsity_and_elimination.ipynb)

# Chapter 1: スパース性、変数消去、Bayes Tree

**SLAM Handbook Section 1.5–1.7** の内容を実装します。

## このNotebookで学ぶこと

1. **スパース構造** — SLAMのヤコビアン・情報行列がなぜスパースか
2. **Cholesky分解とfill-in** — 変数順序がfill-inに与える影響
3. **変数消去アルゴリズム** — Factor GraphからBayes Netへの変換
4. **Bayes Tree** — クリーク構造とインクリメンタル推論の基礎
5. **大規模SLAM問題のシミュレーション** — 100ポーズ+20ランドマークでスパース性を体感

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.sparse import csc_matrix
from scipy.sparse.linalg import splu
import networkx as nx

np.set_printoptions(precision=4, suppress=True)
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

## 3.1 大規模SLAMのシミュレーション

SLAM Handbook Fig. 1.5 のような、ロボットが環境中を移動しながらランドマークを観測する大規模SLAM問題をシミュレートします。

これにより、ヤコビアン行列 $\mathbf{A}$ と情報行列 $\boldsymbol{\Lambda} = \mathbf{A}^\top\mathbf{A}$ の **スパース構造** を観察できます。

In [ ]:
# =============================================================
# 大規模2D SLAMのシミュレーション
# =============================================================
np.random.seed(42)

n_poses = 100       # ポーズ数
n_landmarks = 20    # ランドマーク数
obs_range = 5.0     # 観測可能距離

# ポーズの真値: 楕円軌道
t = np.linspace(0, 2*np.pi, n_poses, endpoint=False)
poses_x = 10 * np.cos(t)
poses_y = 6 * np.sin(t)
poses_th = t + np.pi/2  # 進行方向

# ランドマークの真値: ランダム配置
lm_x = np.random.uniform(-12, 12, n_landmarks)
lm_y = np.random.uniform(-8, 8, n_landmarks)

# 可視性判定: ポーズからランドマークが obs_range 以内なら観測
visibility = []
for i in range(n_poses):
    for j in range(n_landmarks):
        dist = np.sqrt((poses_x[i] - lm_x[j])**2 + (poses_y[i] - lm_y[j])**2)
        if dist < obs_range:
            visibility.append((i, j))

print(f"ポーズ数: {n_poses}, ランドマーク数: {n_landmarks}")
print(f"観測数: {len(visibility)}")
print(f"ファクター数: 1 (prior) + {n_poses-1} (odom) + {len(visibility)} (obs) = "
      f"{1 + n_poses - 1 + len(visibility)}")

# 可視化
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
ax.plot(poses_x, poses_y, 'b.-', markersize=3, alpha=0.5, label='Robot trajectory')
ax.plot(lm_x, lm_y, 'r*', markersize=10, label='Landmarks')

# 観測線を薄く描画
for i, j in visibility[::3]:  # 間引き
    ax.plot([poses_x[i], lm_x[j]], [poses_y[i], lm_y[j]], 'g-', alpha=0.05)

ax.set_aspect('equal')
ax.legend()
ax.set_title(f'Simulated SLAM: {n_poses} poses, {n_landmarks} landmarks, '
             f'{len(visibility)} observations', fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================
# スパースヤコビアンの構造を構築（スパースパターンのみ）
# =============================================================
# 状態ベクトル: [p0(3), p1(3), ..., p99(3), l0(2), l1(2), ..., l19(2)]
n_state = 3 * n_poses + 2 * n_landmarks
n_factors = 3 + 3*(n_poses-1) + len(visibility)  # prior(3) + odom + obs

# ヤコビアンのスパースパターンを記録 (row, col) ペア
rows, cols = [], []
row = 0

# Prior on pose 0 (3 rows)
for dr in range(3):
    for dc in range(3):
        rows.append(row + dr)
        cols.append(dc)
row += 3

# Odometry factors (3 rows each)
for i in range(n_poses - 1):
    for dr in range(3):
        # pose i
        for dc in range(3):
            rows.append(row + dr)
            cols.append(3*i + dc)
        # pose i+1
        for dc in range(3):
            rows.append(row + dr)
            cols.append(3*(i+1) + dc)
    row += 3

# Observation factors (1 row each, bearing)
for pi, lj in visibility:
    # pose pi (3 cols)
    for dc in range(3):
        rows.append(row)
        cols.append(3*pi + dc)
    # landmark lj (2 cols)
    lm_start = 3 * n_poses + 2 * lj
    for dc in range(2):
        rows.append(row)
        cols.append(lm_start + dc)
    row += 1

# スパース行列を構築
data = np.ones(len(rows))
A_sparse = csc_matrix((data, (rows, cols)), shape=(n_factors, n_state))

# 情報行列 Λ = A^T A
Lambda = A_sparse.T @ A_sparse

print(f"ヤコビアン A: {A_sparse.shape}, nnz = {A_sparse.nnz}")
print(f"情報行列 Λ:  {Lambda.shape}, nnz = {Lambda.nnz}")
print(f"A の密度:    {A_sparse.nnz / (A_sparse.shape[0]*A_sparse.shape[1]) * 100:.2f}%")
print(f"Λ の密度:    {Lambda.nnz / (Lambda.shape[0]*Lambda.shape[1]) * 100:.2f}%")

In [ ]:
# =============================================================
# スパースパターンの可視化 (SLAM Handbook Fig. 1.9 風)
# =============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# ヤコビアン A
axes[0].spy(A_sparse, markersize=0.3, color='steelblue')
axes[0].set_title(f'Jacobian A\n{A_sparse.shape}, nnz={A_sparse.nnz}', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Variables (poses | landmarks)')
axes[0].set_ylabel('Factors')
axes[0].axvline(x=3*n_poses - 0.5, color='red', linestyle='--', alpha=0.5)

# 情報行列 Λ = A^T A
axes[1].spy(Lambda, markersize=0.3, color='steelblue')
axes[1].set_title(f'Information Matrix $\\Lambda = A^\\top A$\n{Lambda.shape}, nnz={Lambda.nnz}',
                   fontsize=12, fontweight='bold')
axes[1].axvline(x=3*n_poses - 0.5, color='red', linestyle='--', alpha=0.5)
axes[1].axhline(y=3*n_poses - 0.5, color='red', linestyle='--', alpha=0.5)

# Cholesky因子 R (natural ordering)
Lambda_dense = Lambda.toarray().astype(float)
# 対角要素を大きくして確実に正定値にする（正則化）
Lambda_dense += 1.0 * np.eye(n_state)
R_chol = np.linalg.cholesky(Lambda_dense).T  # 上三角
R_sparse = csc_matrix(R_chol)
R_sparse.data[np.abs(R_sparse.data) < 1e-10] = 0
R_sparse.eliminate_zeros()

axes[2].spy(R_sparse, markersize=0.3, color='steelblue')
axes[2].set_title(f'Cholesky Factor R (natural order)\nnnz={R_sparse.nnz}',
                   fontsize=12, fontweight='bold')
axes[2].axvline(x=3*n_poses - 0.5, color='red', linestyle='--', alpha=0.5)
axes[2].axhline(y=3*n_poses - 0.5, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print(f"→ ランドマーク部分（赤線の右下）にfill-inが多いことが分かります")

## 3.2 変数順序と Fill-in

**Fill-in** = Cholesky分解で元の行列には無かった非ゼロ要素が発生すること。

SLAM Handbook Section 1.5.3: 変数の **順序** が fill-in の量を大きく左右します。

- **Natural ordering** (poses first, then landmarks): ランドマーク部分でfill-inが大きい
- **COLAMD ordering**: ヒューリスティックで fill-in を最小化

fill-inが少ない = メモリ・計算時間が少ない → **大規模SLAMでは変数順序が極めて重要**

In [ ]:
# =============================================================
# 変数順序の影響: Natural vs COLAMD
# =============================================================
from scipy.sparse.csgraph import reverse_cuthill_mckee

Lambda_sp = csc_matrix(Lambda_dense)

# 1. Natural ordering のCholesky (Lambda_dense は既に正則化済み)
R_natural = np.linalg.cholesky(Lambda_dense).T
nnz_natural = np.count_nonzero(np.abs(R_natural) > 1e-10)

# 2. Reverse Cuthill-McKee ordering (fill-in削減ヒューリスティック)
perm_rcm = reverse_cuthill_mckee(Lambda_sp, symmetric_mode=True)
Lambda_rcm = Lambda_dense[np.ix_(perm_rcm, perm_rcm)]
R_rcm = np.linalg.cholesky(Lambda_rcm).T
nnz_rcm = np.count_nonzero(np.abs(R_rcm) > 1e-10)

# 3. ランドマークを先に消去する順序 (landmarks first)
perm_lm_first = np.concatenate([
    np.arange(3*n_poses, n_state),  # landmarks first
    np.arange(3*n_poses),            # then poses
])
Lambda_lmf = Lambda_dense[np.ix_(perm_lm_first, perm_lm_first)]
R_lmf = np.linalg.cholesky(Lambda_lmf).T
nnz_lmf = np.count_nonzero(np.abs(R_lmf) > 1e-10)

# 比較
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, R, title, nnz in zip(axes,
    [R_natural, R_lmf, R_rcm],
    ['Natural\n(poses → landmarks)',
     'Landmarks First\n(landmarks → poses)',
     'Reverse Cuthill-McKee\n(fill-in optimized)'],
    [nnz_natural, nnz_lmf, nnz_rcm]):
    
    ax.spy(csc_matrix(R), markersize=0.3, color='steelblue')
    ax.set_title(f'Cholesky R: {title}\nnnz = {nnz}', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"Fill-in比較:")
print(f"  Natural ordering:      nnz(R) = {nnz_natural}")
print(f"  Landmarks first:       nnz(R) = {nnz_lmf}")
print(f"  Reverse Cuthill-McKee: nnz(R) = {nnz_rcm}")
print(f"\n→ 順序を変えるだけでfill-inが {nnz_natural - nnz_rcm} 要素減少！")

## 3.3 変数消去の可視化

SLAM Handbook Section 1.6, Fig. 1.10 の内容を小さな例で再現します。

変数消去アルゴリズムは Factor Graph を **Bayes Net** に変換します：
1. 変数 $\mathbf{x}_j$ に接続する全ファクターを集める
2. それらの積をQR分解して、条件付き分布 $p(\mathbf{x}_j | \mathbf{s}_j)$ を作る
3. 残りの変数に新しいファクター $\tau(\mathbf{s}_j)$ を追加
4. 繰り返す

これは数値的には **スパースQR分解と等価** です。

In [ ]:
# =============================================================
# 変数消去をグラフで可視化 (Toy SLAM: 3 poses + 2 landmarks)
# =============================================================

def draw_elimination_step(ax, adj, eliminated, current, var_names, title):
    """変数消去の1ステップを描画"""
    G = nx.Graph()
    active = [v for v in var_names if v not in eliminated]
    G.add_nodes_from(active)
    
    for i, vi in enumerate(active):
        for j, vj in enumerate(active):
            if i < j and adj[var_names.index(vi), var_names.index(vj)]:
                G.add_edge(vi, vj)
    
    pos = {
        'ℓ1': (1.5, 2), 'ℓ2': (3.5, 2),
        'p1': (0, 0), 'p2': (2.5, 0), 'p3': (5, 0),
    }
    pos = {k: v for k, v in pos.items() if k in active}
    
    colors = []
    for v in G.nodes():
        if v == current:
            colors.append('lightgray')
        elif v.startswith('ℓ'):
            colors.append('lightyellow')
        else:
            colors.append('lightblue')
    
    nx.draw(G, pos, ax=ax, with_labels=True, node_color=colors,
            node_size=800, font_size=11, font_weight='bold',
            edge_color='black', width=1.5, edgecolors='black')
    ax.set_title(title, fontsize=10, fontweight='bold')

# Toy SLAMの隣接行列 (Fig. 1.3のFactor Graphから導出)
var_names = ['ℓ1', 'ℓ2', 'p1', 'p2', 'p3']
# 隣接関係: 同じファクターに接続 = エッジ
adj = np.zeros((5, 5), dtype=bool)
# p1-p2 (odom), p2-p3 (odom)
adj[2, 3] = adj[3, 2] = True  # p1-p2
adj[3, 4] = adj[4, 3] = True  # p2-p3
# p1-ℓ1, p2-ℓ1 (bearing obs)
adj[0, 2] = adj[2, 0] = True  # ℓ1-p1
adj[0, 3] = adj[3, 0] = True  # ℓ1-p2
# p3-ℓ2 (bearing obs)
adj[1, 4] = adj[4, 1] = True  # ℓ2-p3

# 消去順序: ℓ1, ℓ2, p1, p2, p3
elim_order = ['ℓ1', 'ℓ2', 'p1', 'p2', 'p3']

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes_flat = axes.flatten()

# 初期状態
draw_elimination_step(axes_flat[0], adj, [], None, var_names, 'Initial Factor Graph')

# 各消去ステップ
eliminated = []
adj_current = adj.copy()
for step, var_to_elim in enumerate(elim_order[:-1]):  # 最後の1つは自動
    idx = var_names.index(var_to_elim)
    # neighbors of var_to_elim
    neighbors = [i for i in range(5) if adj_current[idx, i] and var_names[i] not in eliminated]
    # fill-in: 隣接ノード同士を接続
    for ni in range(len(neighbors)):
        for nj in range(ni+1, len(neighbors)):
            adj_current[neighbors[ni], neighbors[nj]] = True
            adj_current[neighbors[nj], neighbors[ni]] = True
    
    eliminated.append(var_to_elim)
    draw_elimination_step(axes_flat[step+1], adj_current, eliminated,
                          var_to_elim, var_names,
                          f'Step {step+1}: Eliminate {var_to_elim}')

# 最後
axes_flat[-1].text(0.5, 0.5, 'All eliminated!\n→ Bayes Net complete',
                    ha='center', va='center', fontsize=14,
                    transform=axes_flat[-1].transAxes)
axes_flat[-1].axis('off')
axes_flat[-1].set_title('Done', fontsize=10, fontweight='bold')

fig.suptitle('Variable Elimination (SLAM Handbook Fig. 1.10)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("→ ℓ1を消去するとp1-p2間にfill-inエッジが生成（既にあった）")
print("→ 変数消去 = グラフの辺を追加しながらノードを除去する操作")

## 3.4 Bayes Tree の構造

SLAM Handbook Section 1.7: 変数消去の結果得られるBayes Netの **クリーク構造** を木で表したものが **Bayes Tree** です。

- 各ノード = クリーク（frontal変数 + separator変数）
- 親子関係 = 条件付き独立性
- **iSAM (incremental Smoothing and Mapping)** はBayes Treeを逐次更新するアルゴリズム

Bayes Treeの核心的なアイデア:
- 新しい計測が追加されたとき、**木の一部だけ** を再計算すれば良い
- 影響を受けるのは、新しいファクターの変数からルートまでのパスのみ

In [ ]:
# =============================================================
# Bayes Tree の可視化 (Toy SLAM, SLAM Handbook Fig. 1.13)
# =============================================================

fig, ax = plt.subplots(1, 1, figsize=(8, 6))

# Bayes Tree for the toy example (elimination order: ℓ1, ℓ2, p1, p2, p3)
BT = nx.DiGraph()

# クリーク定義: frontal : separator
cliques = {
    'p2, p3': None,           # root (no separator)
    'ℓ1, p1 : p2': 'p2, p3', # child, separator = p2
    'ℓ2 : p3': 'p2, p3',     # child, separator = p3
}

for clique, parent in cliques.items():
    BT.add_node(clique)
    if parent:
        BT.add_edge(parent, clique)

pos = {
    'p2, p3': (2.5, 2),
    'ℓ1, p1 : p2': (1, 0),
    'ℓ2 : p3': (4, 0),
}

# ノードの色
colors = ['#90CAF9', '#A5D6A7', '#EF9A9A']

nx.draw(BT, pos, ax=ax, with_labels=True, node_color=colors,
        node_size=3000, font_size=10, font_weight='bold',
        arrows=True, arrowsize=20, arrowstyle='-|>',
        edge_color='gray', width=2, node_shape='s',
        edgecolors='black', linewidths=2)

ax.set_title('Bayes Tree (SLAM Handbook Fig. 1.13)\n'
             'Format: frontal variables : separator', 
             fontsize=13, fontweight='bold')

# 凡例
ax.text(0.02, 0.98, 'Root clique: p(p2, p3)\n'
        'Green child: p(ℓ1, p1 | p2)\n'
        'Red child: p(ℓ2 | p3)',
        transform=ax.transAxes, fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

print("=== Bayes Tree の意味 ===")
print("• Root: p(p2, p3) — 最後に消去される変数群")
print("• 左子: p(ℓ1, p1 | p2) — ℓ1, p1 はp2の条件のもとで独立")
print("• 右子: p(ℓ2 | p3) — ℓ2 はp3にのみ依存")
print()
print("→ 新しい観測が追加されたとき、影響を受けるのはルートまでのパスのみ")
print("→ これがiSAMの効率性の源泉")

## 3.5 スケーラビリティの実験

問題サイズを変えて、Dense vs Sparse の解法時間を比較します。

In [ ]:
# =============================================================
# Dense vs Sparse Cholesky の計算時間比較
# =============================================================
import time

sizes = [50, 100, 200, 500, 1000]
times_dense = []
times_sparse = []

for n in sizes:
    # 帯行列（SLAMライクなスパースパターン）を生成 — 確実に正定値
    A_test = np.zeros((n, n))
    np.fill_diagonal(A_test, 6.0)  # 対角優位にして正定値を保証
    for i in range(n-1):
        A_test[i, i+1] = A_test[i+1, i] = -1.0
    # ランダムに遠くの変数もつなぐ（ランドマーク観測の模擬）
    for _ in range(n // 10):
        i, j = sorted(np.random.choice(n, 2, replace=False))
        A_test[i, j] = A_test[j, i] = -0.3
    
    A_test_sparse = csc_matrix(A_test)
    
    # Dense Cholesky
    t0 = time.perf_counter()
    for _ in range(3):
        np.linalg.cholesky(A_test)
    times_dense.append((time.perf_counter() - t0) / 3)
    
    # Sparse LU (≈ Cholesky for SPD)
    t0 = time.perf_counter()
    for _ in range(3):
        splu(A_test_sparse)
    times_sparse.append((time.perf_counter() - t0) / 3)

fig, ax = plt.subplots(1, 1, figsize=(8, 5))
ax.loglog(sizes, times_dense, 'ro-', markersize=8, label='Dense Cholesky', linewidth=2)
ax.loglog(sizes, times_sparse, 'bs-', markersize=8, label='Sparse LU', linewidth=2)
ax.set_xlabel('Matrix size n')
ax.set_ylabel('Time (seconds)')
ax.set_title('Dense vs Sparse Factorization', fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

print("→ スパース解法は問題サイズが大きくなるほど有利")
print("→ 実際のSLAM (n > 10000) ではスパース解法が必須")

## 3.6 演習問題

### 演習1: 消去順序の比較
Toy SLAM例で消去順序を `[p1, p2, p3, ℓ1, ℓ2]`（ポーズから先に消去）に変えてみましょう。fill-in はどう変わりますか？

### 演習2: ループクロージャの影響
100ポーズのシミュレーションに p0-p50 間のループクロージャファクターを追加して、情報行列のスパースパターンがどう変わるか観察しましょう。

### 演習3: Schur Complement
ランドマーク変数をまず消去（Schur complement）してポーズだけの問題にする手法を実装してみましょう。Visual SLAMで広く使われるテクニックです。

## まとめ

- SLAMのヤコビアン・情報行列は **スパース** — Factor Graphの構造を反映
- **変数順序** がfill-inと計算コストを大きく左右する（COLAMD等のヒューリスティック）
- **変数消去** = Factor Graph → Bayes Net の変換 = スパースQR/Cholesky分解
- **Bayes Tree** = Bayes Netのクリーク構造を木で表現 → インクリメンタル更新(iSAM)の基礎
- 大規模SLAMでは **スパース解法が必須** — Dense解法の $O(n^3)$ vs Sparse解法の $O(n)$〜$O(n^{1.5})$

### Chapter 1 完了！
次は **Chapter 2: Advanced State Variable Representations** — Lie群、回転の表現、マニフォールド上の最適化に進みます。